<a href="https://colab.research.google.com/github/Prajwala15/MLProject/blob/master/updated_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content

!rm -rf textvqa

!git clone https://github.com/Prajwala15/MLProject.git textvqa

%cd textvqa

!ls

/content
Cloning into 'textvqa'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 2), reused 30 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 68.69 KiB | 1.53 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
!pip install -r requirements.txt
!pip install easyocr pytesseract opencv-python

In [ ]:
%cd /content/textvqa/data

!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_test.json

!ls -lh

In [ ]:
!find data -name "*.jpg" | head

In [ ]:
%cd /content/textvqa/data

!unzip train_val_images_1.zip
!unzip test_images.zip

In [ ]:
%%bash

cd /content/textvqa
sed -i 's|data/train_images|data/train_val_images_1|g' configs/default.yaml

In [ ]:
%cd /content/textvqa

In [ ]:
!grep -A5 "images:" configs/default.yaml

**Run easyOCR**

In [ ]:
# Re-run the OCR script to generate the corrected JSON file
!rm -f outputs/ocr_cache/val_easyocr.json

!python scripts/01_run_ocr.py \
    --split val \
    --engine easyocr \
    --limit 100

In [ ]:
# Now, re-load the val_easyocr.json file to verify the fix
import json

with open("outputs/ocr_cache/val_easyocr.json") as f:
    data = json.load(f)

for img_id, tokens in data.items():
    if len(tokens) > 0:
        print("Image ID:", img_id)
        print(tokens[:5])
        break

print("Successfully loaded val_easyocr.json!")

In [ ]:
!rm -f outputs/ocr_cache/val_easyocr.json

!python scripts/01_run_ocr.py \
    --split val \
    --engine easyocr \
    --limit 100

**View OCR output**

In [ ]:
import json

with open("outputs/ocr_cache/val_easyocr.json") as f:
    data = json.load(f)

for img_id, tokens in data.items():
    if len(tokens) > 0:
        print("Image ID:", img_id)
        print(tokens[:5])
        break

**OCR Confidence Analysis**

In [ ]:
import json
import numpy as np

with open("outputs/ocr_cache/val_easyocr.json") as f:
    data = json.load(f)

conf = []

for tokens in data.values():
    for t in tokens:
        conf.append(t["conf"])

print("Average confidence:", np.mean(conf))

**Low-Confidence Error Analysis**

In [ ]:
for img_id, tokens in data.items():
    for t in tokens:
        if t["conf"] < 0.6:
            print(img_id, t)

**Image Preprocessing**

In [ ]:
all_images = !find data/train_val_images_1 -name "*.jpg"

print("Total images:", len(all_images))


In [ ]:
import cv2
import os

def preprocess(img_path):
    img = cv2.imread(img_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    denoise = cv2.fastNlMeansDenoising(gray)

    enhanced = cv2.equalizeHist(denoise)

    return enhanced

processed_count = 0

for img_path in all_images:
    processed = preprocess(img_path)
    processed_count += 1

print("Processed images:", processed_count)

In [ ]:
all_images = !find data/train_val_images_1 -name "*.jpg"

print("Total images:", len(all_images))

**Save processed images**

In [ ]:
import cv2
import os

for img_path in all_images:

    processed = preprocess(img_path)

    filename = os.path.basename(img_path)

    save_path = os.path.join(
        "data/preprocessed_images",
        filename
    )

    cv2.imwrite(save_path, processed)

print("Saved", len(all_images), "preprocessed images")

**Extract text, confidence, and bounding boxes**

In [ ]:
ocr_tokens = []

for img_path, results_list in data.items():

    for item in results_list:
        bbox = item['bbox']
        text = item['text']
        conf = item['conf']

        ocr_tokens.append({
            "image": img_path,
            "text": text,
            "confidence": float(conf),
            "bbox": bbox
        })

print("Total OCR tokens:", len(ocr_tokens))

**Save OCR Results**

In [ ]:
import json

with open("outputs/all_ocr_results.json", "w") as f:
    json.dump(ocr_tokens, f, indent=2)

print("Saved OCR results")

**OCR + Question Fusion**

In [ ]:
import json

question = "What is written on the sign?"

with open("outputs/ocr_cache/val_easyocr.json") as f:
    ocr_data = json.load(f)

img_id = "00831662d2ba731a"

tokens = ocr_data.get(img_id, [])

ocr_text = " ".join([t["text"] for t in tokens])

model_input = (
    "Question: " + question +
    " OCR: " + ocr_text
)

print(model_input)

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY — set Runtime→GPU")